# Look at model coefficients

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

coef_dir = BASE_PATH / "perm_res/coef"

In [ ]:
parent_base = pd.read_parquet(
    BASE_PATH / "perm_res/preds/baseline/parent/coef_df.parquet"
)
self_base = pd.read_parquet(BASE_PATH / "perm_res/preds/baseline/self/coef_df.parquet")

Create tables

In [ ]:
def get_pivot(df, agg_type, llama, desc_seg, alpha, num_layers):
    df["repeat_split_id"] = df["repeat"].astype(str) + "_" + df["split_id"].astype(str)
    sub_df = df[
        (df["agg_type"] == agg_type)
        & (df["llama_model"] == llama)
        & (df["desc_seg"] == desc_seg)
        & (df["alpha"] == alpha)
        & (df["num_layers"] == num_layers)
    ][["repeat_split_id", "stage", "feature", "coef"]]
    sub_df = sub_df.reset_index(drop=True)
    sub_df["stage_feat"] = sub_df["stage"] + "_" + sub_df["feature"]  # y-axis
    # Pivot: rows = stage_feat, columns = repeat, values = avg_coef
    pivot = sub_df.pivot(index="stage_feat", columns="repeat_split_id", values="coef")
    return pivot


def plot_heatmap(pivot):
    ## PLOT
    cmap = LinearSegmentedColormap.from_list(
        "blue_white_green",
        ["#08306b", "white", "#00441b"],  # dark blue, white, dark green
    )
    abs_max = pivot.abs().max().max()
    fig, ax = plt.subplots(figsize=(len(pivot.columns) * 0.8 + 1, 12))

    sns.heatmap(
        pivot,
        annot=True,  # show values in cells
        fmt=".2f",  # format to 2 decimal places
        cmap=cmap,
        center=0,
        # vmin=-abs_max,
        # vmax=abs_max,
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        ax=ax,
    )

    ax.set_title("avg_val by Repeat and Feature")
    ax.set_xlabel("Repeat")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

In [ ]:
impact_type = "parent"
df = self_base.copy() if impact_type == "self" else parent_base.copy()
pivot = get_pivot(
    df=df,
    agg_type="proba-limited",
    llama="Llama-3.3-70B-Instruct",
    desc_seg="combined",
    alpha="20.0",
    num_layers=2,
)

plot_heatmap(pivot)